In [1]:
import pandas as pd
import numpy as np

In [3]:
taxonomy = pd.read_csv("data/gtdb_taxonomy.tsv", sep="\t", header=None)
genome_id = taxonomy.iloc[:,0].values

### pick columns: checkm2_completeness, checkm2_contamination, ncbi_genome_category

In [2]:
file_path = "data/ar53_metadata_r220.tsv.gz"
df_ar = pd.read_csv(file_path, sep='\t', compression='gzip', index_col=0)
df_ar.shape

(12477, 112)

In [4]:
df_ar = df_ar.loc[np.intersect1d(genome_id, df_ar.index.values)]

In [5]:
df_ar = df_ar.loc[(df_ar.checkm2_completeness > 90) & (df_ar.checkm2_contamination < 5) & (df_ar.ncbi_genome_category != "derived from metagenome")]

In [6]:
df_ar.shape

(816, 112)

In [7]:
file_path = "data/bac120_metadata_r220.tsv.gz"
df_bac = pd.read_csv(file_path, sep='\t', compression='gzip', index_col=0)
df_bac.shape

(584382, 112)

In [8]:
df_bac = df_bac.loc[np.intersect1d(genome_id, df_bac.index.values)]

In [9]:
df_bac = df_bac.loc[(df_bac.checkm2_completeness > 90) & (df_bac.checkm2_contamination < 5) & (df_bac.ncbi_genome_category != "derived from metagenome")]

In [11]:
GTDP_high_quality_genome = pd.concat([df_ar[["checkm2_completeness", "checkm2_contamination", "ncbi_genome_category"]],
                                      df_bac[["checkm2_completeness", "checkm2_contamination", "ncbi_genome_category"]]], axis=0)
GTDP_high_quality_genome.index = [i.split('_', 1)[1] for i in GTDP_high_quality_genome.index.values]

In [12]:
GTDP_high_quality_genome.shape

(29323, 3)

In [13]:
GTDP_high_quality_genome.head()

,checkm2_completeness,checkm2_contamination,ncbi_genome_category
GCA_000008085.1,98.72,0.38,none
GCA_000016605.1,99.99,0.38,none
GCA_000145985.1,99.84,0.24,none
GCA_000204585.1,97.64,0.28,derived from single cell
GCA_000224475.1,99.95,0.61,none


In [14]:
patric_df = pd.read_csv("data/ncbi_dataset.tsv", sep="\t")

In [15]:
inter_id = np.intersect1d(patric_df["Assembly Accession"], GTDP_high_quality_genome.index.values)

In [16]:
patric_df_move = patric_df.loc[[i in inter_id for i in patric_df["Assembly Accession"]]]

In [17]:
strain_name = patric_df_move["Organism Infraspecific Names Strain"].values

In [18]:
patric_df = patric_df.loc[[i not in strain_name for i in patric_df["Organism Infraspecific Names Strain"].values]]

In [19]:
patric_df = patric_df.drop_duplicates(subset=['Organism Infraspecific Names Strain'])

In [20]:
patric_df.shape

(1453, 16)

In [21]:
patric_df = patric_df.loc[(patric_df['CheckM completeness'] > 90) & (patric_df["CheckM contamination"] < 5)]

In [22]:
patric_df.shape

(793, 16)

In [23]:
hight_quality_genome = patric_df["Assembly Accession"].tolist() + GTDP_high_quality_genome.index.tolist()

In [24]:
len(hight_quality_genome)

30116

In [25]:
hight_quality_genome = pd.DataFrame({"Accession":hight_quality_genome})

In [26]:
hight_quality_genome.to_csv("data/high_quality_genome.tsv", index=None)